In [ ]:
#tool calling : is a method that an  LLM to use external tools or system to perform tasks it cannot do it itself

In [ ]:
#LLM advantages :it is good at  1.Generation 2.Understanding
# LLM disadvantages :
# 1.fetch data from DATABASE
# 2.news of the today
# 3.Updated data (temp at the movement) 

In [ ]:
#tool calling helps you to overcome the disadvantages
#tool calling is nothing but python function


In [ ]:
#how actually the tool works with LLM
#1. create tools
#2.Tool schema in JSON format (it is just a way to explain what tools and how they acheive things)
#2.1 .what tools
#2.2 Detail description of every tool
#2.3 parameters  
#3. LLM call --> prompt+ Tool Schema
#4 LLM cannot run a function it just select the tool, manually we run the function


In [1]:
import ollama

In [2]:
inventory_db ={
    "laptop":{"stock":5,"base_prise":1200},
    "monitor":{"stock":0,"base_price":300},
    "keyboard":{"stock":25,"base_price":80}
}

In [3]:
def check_inventory(product_name):
    product_name = product_name.lower()
    if product_name in inventory_db:
        return inventory_db[product_name]
    return {"stock":0,"base_price":None}

def calculate_loyalty_discount(base_price,years_as_customer):
    discount = min(years_as_customer*0.05,0.30)
    final_prize = base_price * (1-discount)
    return round(final_prize,2)

In [4]:
available_fuctions = {
    'check_inventory' : check_inventory,
    'calculate_loyalty_discount' :calculate_loyalty_discount,
}

In [7]:
tools = [
    {
        "type":"funtion",
        "function":{
            "name":"check_inventory",
            "description":"Get stock and price for a product",
            "parameters":{
                "type":"object",
                "properties":{
                    "product_name":{"type":"string"}
                },
                "required":["product_name"]
            }
        }
    },
    {
        "type":"funtion",
            "function":{
                 "name":"calculate_loyalty_discount",
                "description":"Get stock and price for a product",
                "parameters":{
                     "type":"object",
                     "properties":{
                         "product_name":{"type":"string"}
                    },
                    "required":["product_name"]
            }
        }
    }
]

In [8]:
tools

[{'type': 'funtion',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string'}},
    'required': ['product_name']}}},
 {'type': 'funtion',
  'function': {'name': 'calculate_loyalty_discount',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string'}},
    'required': ['product_name']}}}]

In [16]:
message = [
    {
        "role":"user",
        "content":"I want to buy a laptop . can you check stock?"
    }
]

responce = ollama.chat(
    model="gemma4:12b",
    messages=message,
    tools=tools
)
print(responce)

model='gemma4:12b' created_at='2026-07-28T17:15:47.6813655Z' done=True done_reason='stop' total_duration=53856616300 load_duration=1779260700 prompt_eval_count=127 prompt_eval_duration=1636511000 eval_count=173 eval_duration=50027814000 message=Message(role='assistant', content='', thinking='The user wants to know the stock status of a "laptop". I should look for a tool that provides information about product availability. The `check_inventory` tool takes a `product_name` as an argument and returns stock and price details. This is the appropriate tool to use.\n\n1.  **Identify the user\'s intent:** The user wants to know if "laptop" is in stock.\n2.  **Match with available tools:** `check_inventory(product_name: str)` is designed for this purpose.\n3.  **Extract parameters:** The product name provided by the user is "laptop".\n4.  **Formulate tool call:** Call `check_inventory(product_name="laptop")`.', images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='check_inv

In [19]:
tool_calls = responce['message'].get('tool_calls')

if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call['function']['name']
        tool_args = tool_call['function']['arguments']

        function_to_call = available_fuctions[tool_name]
        result = function_to_call(**tool_args)

        message.append(responce['message'])

        message.append({
            'role':'tool',
            'content':str(result)
        })

final_responce = ollama.chat(
    model ='gemma4:12b',
    messages = message,
)

In [20]:
print(final_responce['message']['content'])

Yes, we have 5 laptops currently in stock. The base price for each unit is $1,200.
